In [ ]:
import os
import json
from datasets import load_dataset
from tqdm import tqdm

def process_data_from_iterator(ds_iterator, output_file, num_samples, tag="Train"):
    print(f"--> Copying [{tag}] , target: {num_samples} ..")
    
    count = 0
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w", encoding="utf-8") as f:
        pbar = tqdm(total=num_samples, desc=f"{tag} Progress")
        
        for item in ds_iterator:
            text = item.get("text", "")
            
            if len(text) < 10000: 
                continue
                
            clean_text = text[:30000] 
            f.write(json.dumps({"text": clean_text}, ensure_ascii=False) + "\n")
            
            count += 1
            pbar.update(1)
            
            if count >= num_samples:
                break
        
        pbar.close()
    print(f"--> [{tag}] Done  saved {count} samples to {output_file}")

if __name__ == "__main__":
    output_dir = "Your/Output/Path"
    
    print("--> Copying streaming dataset...")
    # 1. 在主函数只加载一次
    dataset = load_dataset(
        "HuggingFaceFW/fineweb-edu", 
        name="sample-10BT", 
        split="train", 
        streaming=True
    )

    ds_iterator = iter(dataset)

    process_data_from_iterator(
        ds_iterator, 
        os.path.join(output_dir, "phase1_train.jsonl"), 
        num_samples=1000,
        tag="Train_1"
    )


    process_data_from_iterator(
        ds_iterator, 
        os.path.join(output_dir, "phase1_eval.jsonl"), 
        num_samples=200,
        tag="Eval"
    )

    process_data_from_iterator(
        ds_iterator, 
        os.path.join(output_dir, "phase2_train.jsonl"), 
        num_samples=10000,
        tag="Train_2"
    )


    